# A.X-Encoder 학습 곡선 — EPOCHS를 몇으로 할지 정한다

**이 노트북은 계측 전용이다. 모델을 저장하지도, 배포하지도 않는다.**
목적은 딱 하나 — 검증 성능이 어느 epoch에서 정점을 찍는지 그래프로 확인하는 것.

## 왜 따로 만들었나

원본 `AX_Encoder_평가_v3_1000행_최종.ipynb`의 `train_one_fold`는 **epoch별 학습
loss만** 남기고, 검증은 학습이 다 끝난 뒤 딱 한 번 한다. 그래서 과적합 진단 곡선
(학습 loss와 검증 loss가 갈라지는 그림)을 그릴 재료가 없다.

여기서는 학습 루프 안에서 25스텝마다 검증셋을 재서 그 재료를 만든다.

## 원본에서 뺀 것

이미 결과가 나와 있어 다시 돌릴 이유가 없는 부분은 전부 뺐다.

| 뺀 것 | 원래 시간 | 이미 아는 결과 |
|---|---|---|
| §2 누출 기준선 (TF-IDF) | 2분 | response-only 0.391 / history-only 0.294 |
| §4 단일 fold 점검 | 5분 | 아래 곡선이 대체한다 |
| §5 전체 교차검증 | 23분 | OOF 0.854 |
| §6~§7-b 판정·지표·기저율 | — | swapped 0.469 / empty 0.485 / F1 최대 thr 0.50 |
| §9 배포용 최종 모델 | 10분 | **원본 노트북에서 한다** (맨 아래 참조) |

남긴 건 세 셀뿐이다 — 환경 확인, 데이터 로드, 모델·하이퍼파라미터.
뒤 셀이 여기서 만든 `df` / `y` / `groups` / `encode` / `load_model` 을 쓴다.

## 실행 순서

위에서부터 차례로. 마지막 셀까지 **약 20~25분** (T4 기준, 16 epoch).


## 1. 환경 확인 — **설치하지 않는다**

In [ ]:
# 설치하지 않는다. Colab 기본 환경에 전부 들어 있다.
# `pip install -U` 는 pandas/numpy를 메이저 업그레이드해 torch 임포트를 깨뜨린다.
def _ver(name):
    try:
        return getattr(__import__(name), "__version__", "?")
    except Exception as e:
        return f"!! {type(e).__name__}: {e}"

vers = {m: _ver(m) for m in ["numpy", "pandas", "sklearn", "torch", "transformers"]}
for m, v in vers.items():
    print(f"{m:14s} {v}")

broken = [m for m, v in vers.items() if str(v).startswith("!!")]
print()
if broken:
    print(f"[환경 손상] {', '.join(broken)} 임포트 실패.")
    print("  → 런타임 → '런타임 연결 해제 및 삭제' 후 새 런타임에서 이 셀부터 다시 실행.")
    print("  → pip install 은 하지 말 것.")
else:
    from packaging.version import Version
    ok = Version(vers["transformers"].split("+")[0]) >= Version("4.48")
    print(f"transformers {vers['transformers']} — ModernBERT 지원 {'OK' if ok else '부족'}")
    if not ok:
        print("  → !pip install -U transformers  (다른 패키지는 건드리지 말 것)")
        print("  → 설치 후 '런타임 → 세션 다시 시작' 하고 이 셀부터 다시")
    import torch
    print(f"CUDA {torch.cuda.is_available()} | "
          f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU 없음 — 런타임 유형을 T4로'}")

## 2. 데이터 · 모델 준비

원본 §1·§3과 같다. 뒤 셀이 여기서 만든 것을 쓴다.

In [ ]:
CSV_PATH  = "training_dataset_v3_1000_simple.csv"
GROUP_BY  = "pair"   # "pair" = 프로젝트 규칙(pair_id 단위) | "strict" = pair+history 연결 성분

import os, numpy as np, pandas as pd
if not os.path.exists(CSV_PATH):
    from google.colab import files
    CSV_PATH = list(files.upload().keys())[0]

df = pd.read_csv(CSV_PATH)
assert not ({"pair_id","history","response","label"} - set(df.columns)), f"컬럼 누락: {list(df.columns)}"
df = df.dropna(subset=["history","response","label"]).reset_index(drop=True)
df["history"]  = df["history"].astype(str)
df["response"] = df["response"].astype(str)

# --- 화자 익명화 (운영 경로와 형식을 맞춘다) --------------------------------
# 학습 데이터의 history는 "선임:" "나:" "팀장:" 같은 실제 호칭을 쓴다(66종).
# 그런데 웹앱은 FR-6.2에 따라 화자를 A/B/C로 익명화해서 모델에 넣는다.
# 그대로 학습하면 배포 시 **학습에서 한 번도 못 본 형식**이 들어간다.
# anonymize.label_speakers 와 같은 규칙(등장 순서 → A,B,C…)으로 미리 바꾼다.
import re
ANONYMIZE_SPEAKERS = True

def _anonymize(h: str) -> str:
    AL = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    seen, out = {}, []
    for line in str(h).replace("\r\n", "\n").split("\n"):
        mo = re.match(r"^([^:]{1,10}):\s*(.*)$", line)
        if not mo:
            out.append(line); continue
        who, txt = mo.group(1), mo.group(2)
        if who not in seen:
            seen[who] = AL[len(seen)] if len(seen) < 26 else f"S{len(seen)}"
        out.append(f"{seen[who]}: {txt}")
    return "\n".join(out)

if ANONYMIZE_SPEAKERS:
    before = df["history"].iloc[0][:56]
    df["history"] = df["history"].map(_anonymize)
    print("\n[익명화] 화자 호칭 → A/B/C (운영 프롬프트와 동일 규칙)")
    print(f"  전: {before}...")
    print(f"  후: {df['history'].iloc[0][:56]}...")

y = (df["label"].astype(str).str.strip() == "부적절").astype(int).values

if GROUP_BY == "strict":
    # pair_id와 history를 같은 그룹으로 묶는다(union-find). 같은 방의 여러 응답이
    # 학습·검증에 갈라지는 것까지 막는 더 엄격한 분할.
    par = {}
    def find(x):
        par.setdefault(x, x)
        while par[x] != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: par[ra] = rb
    for p, h in zip(df["pair_id"].astype(str), df["history"]):
        union("P:" + p, "H:" + h)
    groups = np.array([find("P:" + p) for p in df["pair_id"].astype(str)])
else:
    groups = df["pair_id"].astype(str).values

print(f"{CSV_PATH} | {len(df)}행 | 부적절 {y.sum()} / 적절 {(1-y).sum()}")
print(f"그룹 방식 '{GROUP_BY}' → 그룹 {len(set(groups))}개 (평균 {len(df)/len(set(groups)):.1f}행)")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "skt/A.X-Encoder-base"
LR = 5e-5
BATCH = 8

# 이 노트북은 EPOCHS를 쓰지 않는다 — epoch 수는 아래 곡선 셀의
# PROBE_EPOCHS가 정한다. 행 수에 따라 달라지는 건 토큰 길이와 seed 뿐이다.
if len(df) < 300:
    SEEDS, MAX_LEN = [42, 7, 2026], 256
else:
    SEEDS, MAX_LEN = [42], 384

steps_per_epoch = int(np.ceil(len(df) * 0.8 / BATCH))
print(f"{len(df)}행 → SEEDS={SEEDS} | MAX_LEN={MAX_LEN} | LR={LR} | BATCH={BATCH}")
print(f"fold당 {steps_per_epoch}스텝/epoch — 총 스텝 수는 PROBE_EPOCHS가 정한다")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_model():
    try:
        return AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2, attn_implementation="sdpa").to(device)
    except Exception:
        return AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2).to(device)

def encode(h, r):
    return tokenizer(list(h), list(r), truncation="only_first",
                     max_length=MAX_LEN, padding=True, return_tensors="pt")

lens = [len(tokenizer(h, r)["input_ids"]) for h, r in zip(df["history"][:400], df["response"][:400])]
print(f"\n토큰 길이(표본 400) 중앙 {int(np.median(lens))} | p95 {int(np.percentile(lens,95))} | 최대 {max(lens)}")
print(f"MAX_LEN={MAX_LEN} → {'전부 들어간다' if max(lens) <= MAX_LEN else '일부 잘림 (history 앞부분부터)'}")

## 3. 학습 곡선 측정

`PROBE_EPOCHS = 16` 으로 한 fold를 길게 돌리면서 **25스텝마다** 세 가지를 잰다.

| 재는 것 | 왜 |
|---|---|
| 검증 loss | **주 지표.** 최저점이 곧 최적 epoch |
| 검증 정확도 | 발표용. loss 최저점과 어긋날 수 있다 |
| swapped 정확도 | 이 프로젝트 고유. 문맥을 얼마나 보는지 |

**swapped를 같이 재는 이유** — 정확도가 올라도 `검증 − swapped` 하락폭이 줄어들면
문맥이 아니라 응답 표면에 기대기 시작했다는 뜻이다. 이 프로젝트에서는 정확도보다
중요한 신호라, 하락폭이 무너지는 epoch은 정확도가 높아도 고르지 않는다.

### 읽을 때 주의할 것

학습률이 마지막에 0으로 수렴하도록 짜여 있어서 **후반 epoch은 모델이 거의 안
움직인다.** 곡선 꼬리가 평평해 보이는 건 과적합이 없어서가 아니라 학습이 멈춰서일
수 있다. 그래서 이 곡선은 정밀 측정이 아니라 **후보를 좁히는 용도**로 쓴다.
최종 판단은 원본 노트북에서 후보 epoch으로 5-fold를 돌려
이미 갖고 있는 4 epoch 결과(OOF 0.854)와 직접 비교해서 내린다.


In [ ]:
# =====================================================================
# 학습 곡선 측정 — 25스텝마다 검증셋을 잰다
# =====================================================================
import time
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedGroupKFold

PROBE_EPOCHS = 16   # 꺾이는 지점을 보려면 현재 설정(4)보다 넉넉해야 한다
EVAL_EVERY   = 25   # 몇 스텝마다 검증할지. 작을수록 곡선이 촘촘하고 느리다
SEED_CURVE   = SEEDS[0]

cvc = StratifiedGroupKFold(5, shuffle=True, random_state=SEED_CURVE)
tr_idx, va_idx = next(iter(cvc.split(df, y, groups)))

torch.manual_seed(SEED_CURVE); np.random.seed(SEED_CURVE)
model = load_model()

enc_tr = encode(df["history"].values[tr_idx], df["response"].values[tr_idx])
dl = DataLoader(
    TensorDataset(enc_tr["input_ids"], enc_tr["attention_mask"],
                  torch.tensor(y[tr_idx], dtype=torch.long)),
    batch_size=BATCH, shuffle=True)

h_va = df["history"].values[va_idx]
r_va = df["response"].values[va_idx]
h_sw = np.roll(h_va, 1)                       # §6의 swapped와 같은 방식
lab_va = torch.tensor(y[va_idx], dtype=torch.long)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total = len(dl) * PROBE_EPOCHS
sch = get_linear_schedule_with_warmup(opt, max(1, int(total * 0.1)), total)
amp = torch.bfloat16 if USE_BF16 else torch.float32

steps_total = len(dl) * PROBE_EPOCHS
print(f"학습 {len(tr_idx)}행 / 검증 {len(va_idx)}행")
print(f"{PROBE_EPOCHS} epoch × {len(dl)}스텝 = {steps_total}스텝 · "
      f"검증 {steps_total // EVAL_EVERY}회\n")


@torch.no_grad()
def _logits(h, r):
    out = []
    for i in range(0, len(r), 64):
        e = encode(h[i:i+64], r[i:i+64])
        with torch.autocast("cuda", dtype=amp, enabled=USE_BF16):
            lg = model(input_ids=e["input_ids"].to(device),
                       attention_mask=e["attention_mask"].to(device)).logits
        out.append(lg.float().cpu())
    return torch.cat(out)


@torch.no_grad()
def measure():
    """검증 loss / 검증 정확도 / swapped 정확도"""
    model.eval()
    lg = _logits(h_va, r_va)
    vloss = F.cross_entropy(lg, lab_va).item()
    vacc = (lg.argmax(-1) == lab_va).float().mean().item()
    sacc = (_logits(h_sw, r_va).argmax(-1) == lab_va).float().mean().item()
    model.train()
    return vloss, vacc, sacc


rows = []
vl, va, sa = measure()                       # 학습 전 출발점
rows.append({"step": 0, "epoch": 0.0, "train_loss": np.nan,
             "val_loss": vl, "val_acc": va, "swap_acc": sa})

print(f"{'step':>5} {'epoch':>6} {'학습loss':>8} {'검증loss':>8} "
      f"{'검증acc':>8} {'swap':>7} {'하락폭':>7} {'경과':>7}")

step, window, t0 = 0, [], time.time()
model.train()
for ep in range(PROBE_EPOCHS):
    for ids, mask, lab in dl:
        ids, mask, lab = ids.to(device), mask.to(device), lab.to(device)
        with torch.autocast("cuda", dtype=amp, enabled=USE_BF16):
            loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step(); opt.zero_grad()

        window.append(loss.item()); step += 1
        if step % EVAL_EVERY == 0:
            vl, va, sa = measure()
            tl = float(np.mean(window)); window = []
            rows.append({"step": step, "epoch": step / len(dl), "train_loss": tl,
                         "val_loss": vl, "val_acc": va, "swap_acc": sa})
            print(f"{step:5d} {step/len(dl):6.2f} {tl:8.4f} {vl:8.4f} "
                  f"{va:8.3f} {sa:7.3f} {va-sa:+7.3f} {time.time()-t0:6.0f}s")

curve = pd.DataFrame(rows)
curve["gap"] = curve["val_acc"] - curve["swap_acc"]
curve.to_csv("learning_curve.csv", index=False)

del model; torch.cuda.empty_cache()

# ---------------------------------------------------------------- 판정
c = curve.dropna(subset=["train_loss"]).reset_index(drop=True)
i_loss = int(c["val_loss"].idxmin())
i_acc  = int(c["val_acc"].idxmax())
i_gap  = int(c["gap"].idxmax())

print("\n" + "=" * 70)
print(f"검증 loss 최저   : epoch {c['epoch'][i_loss]:5.2f}  "
      f"loss {c['val_loss'][i_loss]:.4f}  acc {c['val_acc'][i_loss]:.3f}  "
      f"하락폭 {c['gap'][i_loss]:+.3f}")
print(f"검증 정확도 정점 : epoch {c['epoch'][i_acc]:5.2f}  "
      f"acc {c['val_acc'][i_acc]:.3f}  하락폭 {c['gap'][i_acc]:+.3f}")
print(f"하락폭 최대      : epoch {c['epoch'][i_gap]:5.2f}  "
      f"하락폭 {c['gap'][i_gap]:+.3f}  acc {c['val_acc'][i_gap]:.3f}")
print("-" * 70)

tail = c["val_loss"].iloc[-1] - c["val_loss"][i_loss]
if i_loss >= len(c) - 2:
    print(f"[정점 미검출] 검증 loss가 마지막까지 최저다. {PROBE_EPOCHS} epoch으로도 "
          f"부족하다.\n  → PROBE_EPOCHS를 24~32로 올려 다시 볼 것.")
elif tail > 0.05:
    print(f"[과적합 확인] 최저점 이후 검증 loss가 {tail:+.4f} 올라간다.")
else:
    print(f"[과적합 약함] 최저점 이후 상승폭이 {tail:+.4f}로 작다. "
          f"학습률이 0으로 수렴해 후반이 눌렸을 수 있다.")

cands = sorted({max(1, round(c["epoch"][i])) for i in (i_loss, i_acc, i_gap)})
print(f"\n다음 단계에서 검증할 후보 EPOCHS: {cands}")
print("  → 원본 노트북에서 이 값으로 5-fold를 돌려 4 epoch 결과(OOF 0.854)와 비교")
print("=" * 70)
print(f"\n총 {time.time()-t0:.0f}초 · learning_curve.csv 저장")


In [ ]:
# =====================================================================
# 발표용 그래프 — curve_loss.png / curve_context.png 로 저장된다
# =====================================================================
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Colab에는 한글 폰트가 없다. 한 번만 설치하면 된다.
if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
    subprocess.run("apt-get install -y -qq fonts-nanum", shell=True)
    fm._load_fontmanager(try_read_cache=False)

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASE, SURF = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "font.family": "NanumGothic", "axes.unicode_minus": False,
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
})


def _clean(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(BASE); ax.spines["bottom"].set_color(BASE)
    ax.tick_params(colors=MUTED, labelsize=11, length=0)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color=GRID, lw=1); ax.xaxis.grid(False)


def _title(fig, ax, t, sub, tsize=16.5):
    fig.text(0.0, 1.10, t, transform=ax.transAxes, fontsize=tsize,
             color=INK, fontweight="bold", va="bottom", ha="left")
    fig.text(0.0, 1.03, sub, transform=ax.transAxes, fontsize=11,
             color=MUTED, va="bottom", ha="left")


x  = c["epoch"].values
tr = c["train_loss"].values
vl = c["val_loss"].values

# ---------- 그래프 1: 과적합 진단 (학습 loss vs 검증 loss) ----------
fig, ax = plt.subplots(figsize=(8.8, 5.2))
ax.plot(x, tr, color=BLUE,   lw=2.4, zorder=4, solid_capstyle="round", label="학습 loss")
ax.plot(x, vl, color=ORANGE, lw=2.4, zorder=4, solid_capstyle="round", label="검증 loss")

ax.scatter([x[i_loss]], [vl[i_loss]], s=120, color=ORANGE, ec=SURF, lw=2.5, zorder=6)
ax.axvline(x[i_loss], color=MUTED, lw=1.5, ls=(0, (5, 4)), zorder=2)
ax.annotate(f"검증 loss 최저\nepoch {x[i_loss]:.1f}",
            xy=(x[i_loss], vl[i_loss]), xytext=(12, 24), textcoords="offset points",
            fontsize=11.5, color=INK2, fontweight="bold", zorder=7)

if vl[-1] - vl[i_loss] > 0.03:
    mid = (i_loss + len(x) - 1) // 2
    ax.annotate("여기부터 외우기 시작", xy=(x[mid], vl[mid]),
                xytext=(-16, 32), textcoords="offset points", ha="right",
                color=ORANGE, fontsize=12, fontweight="bold", zorder=7)

ax.set_xlabel("epoch", fontsize=11, color=INK2, labelpad=8)
ax.set_ylabel("loss", fontsize=11, color=INK2, labelpad=10)
ax.set_ylim(bottom=0)
_clean(ax)
ax.legend(frameon=False, fontsize=12, labelcolor=INK2,
          loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
_title(fig, ax, "검증 loss가 다시 올라가는 지점이 과적합 시작점",
       f"A.X-Encoder · 1000행 · fold 1개 · 학습 loss는 끝까지 내려가지만 검증은 아니다")
fig.subplots_adjust(top=0.84)
fig.savefig("curve_loss.png", dpi=200, bbox_inches="tight")
plt.show()

# ---------- 그래프 2: 문맥 의존 (검증 vs swapped) ----------
va_ = c["val_acc"].values
sa_ = c["swap_acc"].values

fig, ax = plt.subplots(figsize=(8.8, 5.2))
ax.fill_between(x, sa_, va_, color=BLUE, alpha=0.10, zorder=2, lw=0)
ax.plot(x, va_, color=BLUE,   lw=2.4, zorder=4, solid_capstyle="round", label="검증 정확도")
ax.plot(x, sa_, color=ORANGE, lw=2.4, zorder=4, solid_capstyle="round", label="swapped (문맥 교체)")

ax.scatter([x[i_gap]], [va_[i_gap]], s=120, color=BLUE, ec=SURF, lw=2.5, zorder=6)
ax.annotate(f"하락폭 최대 {c['gap'][i_gap]:+.3f}\nepoch {x[i_gap]:.1f}",
            xy=(x[i_gap], va_[i_gap]), xytext=(10, 16), textcoords="offset points",
            fontsize=11.5, color=INK2, fontweight="bold", zorder=7)

ax.axhline(0.5, color=MUTED, lw=1.5, ls=(0, (5, 4)), zorder=2)
ax.text(x[-1], 0.508, "찍기 수준 0.5", color=MUTED, fontsize=10, ha="right")

ax.set_xlabel("epoch", fontsize=11, color=INK2, labelpad=8)
ax.set_ylabel("정확도", fontsize=11, color=INK2, labelpad=10)
ax.set_ylim(0.3, 1.0)
_clean(ax)
ax.legend(frameon=False, fontsize=12, labelcolor=INK2,
          loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
_title(fig, ax, "두 선의 간격이 곧 '문맥을 읽은 정도'다",
       "간격이 좁아지면 정확도가 올라도 응답 표면에 기대기 시작한 것 — 그 epoch은 피한다")
fig.subplots_adjust(top=0.84)
fig.savefig("curve_context.png", dpi=200, bbox_inches="tight")
plt.show()

print("저장: curve_loss.png · curve_context.png · learning_curve.csv")


## 다음 단계 — 배포 모델은 원본 노트북에서 만든다

이 노트북은 여기까지다. 모델을 저장하지 않는다.

위에서 나온 **후보 EPOCHS**를 들고 원본
`AX_Encoder_평가_v3_1000행_최종.ipynb` 으로 돌아가서:

1. **셀 7의 자동 분기를 덮어쓴다.** 지금은 행 수를 보고 `EPOCHS = 4`가 되므로,
   그 아래에 `EPOCHS = <후보값>` 한 줄을 추가해 고정한다.
2. **§1부터 §9까지 정주행한다.** 그래야 새 epoch 기준으로
   §5 OOF · §6 절제 실험 · §7 threshold가 전부 다시 계산되고,
   §9가 그 OOF에서 threshold를 뽑아 `context_checker/` 를 저장한다.
3. **4 epoch 결과와 비교한다.** 이미 갖고 있는 값은 OOF 0.854 /
   swapped 하락폭 +0.385 다. 새 epoch이 이걸 넘지 못하면 4를 유지한다.
4. 이긴 쪽의 `context_checker/` 를 내려받아
   `backend/models/context_checker/` 를 교체한다.

> **왜 곡선 도중의 가중치를 그냥 저장하지 않나** — 16 epoch 스케줄로 돌리다
> 중간에 멈춘 모델은 학습률이 아직 남아 있는 어중간한 상태다. 그 epoch 수로
> 처음부터 다시 돌리면 학습률이 0까지 부드럽게 착지해 보통 더 좋다.
> 곡선은 **몇 epoch이 좋은지만 알려주는 계측**이고, 실제 모델은 그 값으로 새로 만든다.

### 발표에 쓸 때

`curve_loss.png` 는 "EPOCHS를 왜 그 값으로 정했는가"의 근거,
`curve_context.png` 는 "더 학습시켜도 문맥을 계속 읽는가"의 근거다.
두 번째가 이 프로젝트에서는 더 중요하다.
